# Intoduction and first steps

This notebook is a tutorial that introduces Eradiate and its approach to setting up and running simulations. It is meant to give a quick and hands on example of how to set up and run a simulation in Eradiate.
For details about specialized experiment types and scene elements, users should refer to the online documentation and the remainder of the tutorial suite.

## Imports

The first thing to do is to import all necessary python modules. Apart from modules like and `matplotlib` this starts with the `eradiate` module

In [ ]:
import matplotlib.pyplot as plt

import eradiate

Once the `eradiate` module is imported, we can set the operational mode and add some aliases to submodules of Eradiate for convenience.

The *mono* mode is designed to perform monochromatic simulations. It uses double floating point precision by default.

The `eradiate.scenes` module holds all classes that descibe the scene elements of an Eradiate scene. We will be instantiating a lot of classes from this module, so this alias is very convenient.

Eradiate uses the `pint` module, which lets users create quantities with units attached and converts those quantities between compatible units. This makes scene creation more convenient as all objects can be described in natural units. For example atmospheric height might be defined in kilometers, while the radius of a leaf in a canopy would rather be defined centimeters.
The Eradiate unit registry lets users attach units to numbers, creating the aforementioned quantities. Accordingly we will be using it heavily and therefore introduce an alias to it.

In [ ]:
eradiate.set_mode("mono")

import eradiate.scenes as ertsc
from eradiate import unit_registry as ureg

## Creating the scene

Run a simulation with Eradiate, an [Experiment](../rst/reference_api/generated/autosummary/eradiate.experiments.Experiment.rst) object must be created. An experiment holds [SceneElement](../rst/reference_api/generated/autosummary/eradiate.scenes.core.SceneElement.rst) objects which define the scene's constituents.

Let's set up a few scene elements and use them to create our experiment.

First we create a surface. In experiments with atmospheres or canopies surfaces can be defined directly through the BSDF and the user does not have to think about the shape or size of the underlying surface object. The horizontal extent of the surface will automatically be adapted to the size of the other scene constituents.
Here we create a surface with lambertian scattering ([LambertianBSDF](../rst/reference_api/generated/autosummary/eradiate.scenes.bsdfs.LambertianBSDF.rst)) and a constant reflectance of 0.5:

In [ ]:
surface_bsdf = ertsc.bsdfs.LambertianBSDF(reflectance=0.5)

Next we define the atmosphere for our simulation. We use the constructor function for the US Standard 1976 atmospheric profile ([MolecularAtmosphere](../rst/reference_api/generated/autosummary/eradiate.scenes.atmosphere.MolecularAtmosphere.rst)).
An atmosphere defined in this way will exhibit scattering and absorption properties as they are defined in the model.

In [ ]:
atmosphere = ertsc.atmosphere.MolecularAtmosphere.ussa_1976()

To illuminate the scene, we are using a [DirectionalIllumination](../rst/reference_api/generated/autosummary/eradiate.scenes.illumination.DirectionalIllumination.rst) object, which can be defined using only a zenith and azimuth angle. In this case the irradiance spectrum will default to the Thuillier 2003 spectrum.

In [ ]:
illumination = ertsc.illumination.DirectionalIllumination(
    zenith=15.0,
    azimuth=0.0,
)

Finally we define the observation. We use a [DistantSensor](../rst/reference_api/generated/autosummary/eradiate.scenes.measure.DistantSensor.rst), which records leaving radiance for the defined observation geometries.

We specify a set of zenith values covering the range from -75° to 75° in 5° steps. By default the list of azimuth values has to be the same length as the list of zenith values as they form pairs. Alternatively one can pass a single azimuth value, which will be the same for all zenith values. The same approach is possible for a constant zenith angle. We choose the azimuth angle to be the same as for the illumination, this way the sensor will cover the principal plane.

The `spectral_cfg` parameter defines the wavelength at which the simulation will be run. If multiple values are passed to this parameter, Eradiate will automatically loop on those wavelengths and the output dataset will contain a wavelength dimension.

The `spp` parameter defines the number of samples which are drawn for each pixel of this sensor.

In [ ]:
measure = ertsc.measure.MultiDistantMeasure.from_viewing_angles(
    id="toa_brf",
    zeniths=np.arange(-75, 76, 5),
    azimuths=0,
    spectral_cfg={
        "wavelengths": [550]
    },
    spp=10000,
)

Now we can set up the [OneDimExperiment](../rst/reference_api/generated/autosummary/eradiate.experiments.OneDimExperiment.rst) object with the scene elements we just created.

In [ ]:
exp = eradiate.experiments.OneDimExperiment(
    surface=surface_bsdf,
     atmosphere=atmosphere,
    illumination=illumination,
    measures=measure,
)

## Running the simulation and visualizing the results

Calling `eradiate.run()` executes the simulation we just defined and returns a xarray Dataset with the results of the simulation. The post-processing pipeline for this experiment class will additionally compute the BRF and BRDF of the simulated scene and add them to the dataset.

In [ ]:
results = eradiate.run(exp)

To visualize the results, we create a matplotlib figure and plot the BRF data, using xarray's builtin plotting methods. Note that the BRF is selected by accessing the `brf` member of the results dataset. By specifying the x-coordinate in the plotting method, we choose which coordinate to plot against.

In [ ]:
fig = plt.figure(figsize=(8,6), dpi=100)
results.brf.plot(x="vza")

## Final remarks

As we have seen, setting up a simple simulation with Eradiate is straight forward. Inclined users can now tweak the parameters of this tutorial, for example increasing the number of observations or the number of samples per pixel.

The tutorial suite contains further examples with detailed explanations of Eradiate's atmospheric simulation capabilities, three dimensional canopies and more.